# CFM Pipeline Test
Load CFM model, pass an image, get concept vector. Verify pipeline works before smoothing experiments.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from pathlib import Path

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.5.1+cu121
CUDA available: False


In [ ]:
# Load CFM Model (CLIP-DINOiser + SAE)
from cfm.arg_parser import get_default_parser
from cfm.utils import common_init, get_img_model
from cfm.cfm import CFM
from dictionary_learning.utils import load_dictionary

# SAE config name (from Kai's directory)
CONFIG_NAME = 'k_12_ef_16_lr_0.0001_mf_[0.008,0.03,0.06,0.12,0.24,0.542]'

# Build args from config.py defaults
parser = get_default_parser()
args = parser.parse_args([])
args.device = 'cuda' if torch.cuda.is_available() else 'cpu'
common_init(args)
args.config_name = CONFIG_NAME

print("Device:", args.device)
print("Image encoder:", args.img_enc_name)
print("SAE checkpoint dir:", args.save_dir_sae_ckpts['img'])

c:\Daten\D_Part\Personal\CFM_smoothing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
# Load Feature Extractor (CLIP-DINOiser)
feature_extractor, preprocess = get_img_model(args)
feature_extractor.eval()
print("CLIP-DINOiser loaded")

In [ ]:
# Load SAE 
sae_base = Path(args.save_dir_sae_ckpts['img']) / args.config_name / 'trainer_0'
print("Looking for SAE at:", sae_base)
print("Exists:", sae_base.exists())

if sae_base.exists():
    autoencoder, ae_config = load_dictionary(str(sae_base), args.device)
    print("SAE loaded")
    print("SAE config:", ae_config)
else:
    # List what's available
    sae_parent = Path(args.save_dir_sae_ckpts['img'])
    print("Available SAE configs:")
    if sae_parent.exists():
        for d in sae_parent.iterdir():
            print(f"  - {d.name}")
    else:
        print(f"  SAE parent dir doesn't exist: {sae_parent}")
        print("  Check config.py paths!")

In [ ]:
# Create CFM model
cfm_model = CFM(
    feature_extractor=feature_extractor,
    autoencoder=autoencoder,
    apply_found=False,
    device=args.device,
)
cfm_model.eval()
print("CFM model ready")

In [ ]:
# Load concept names
concept_name_save_path = os.path.join(
    args.save_dir_sae_ckpts['img'], 
    args.save_suffix, 
    args.config_name, 
    'trainer_0', 
    'concept_names.txt'
)
print("Looking for concept names at:", concept_name_save_path)

if os.path.exists(concept_name_save_path):
    with open(concept_name_save_path, "r") as f:
        concept_names = [line.strip() for line in f.readlines()]
    print(f"Loaded {len(concept_names)} concept names")
    print("First 20:", concept_names[:20])
else:
    concept_names = None
    print("Concept names file not found, using indices instead.")

## Test image through pipeline

In [ ]:
# Load a test image
import urllib.request
test_img_path = "test_image.jpg"
if not os.path.exists(test_img_path):
    url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg"
    urllib.request.urlretrieve(url, test_img_path)
    print("Downloaded test image")

# Option B: Use an image from a dataset
# Load and preprocess
image = Image.open(test_img_path).convert("RGB")
image_tensor = preprocess(image).unsqueeze(0).to(args.device)

print(f"Image size: {image.size}")
print(f"Tensor shape: {image_tensor.shape}")
print(f"Image size: {image.size}")
# Display
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title("Test Image")
plt.axis("off")
plt.show()

In [ ]:
# Get concept vector
with torch.no_grad():
    concept_vector = cfm_model.get_aggregated_concept_activations(image_tensor)  # [1, 8192]
    concept_maps = cfm_model.get_concept_activation_map(image_tensor)  # [1, 8192, 28, 28]

print("=" * 60)
print("CONCEPT VECTOR (image-level, max-pooled)")
print("=" * 60)
print(f"Shape: {concept_vector.shape}")
print(f"Non-zero concepts: {(concept_vector[0] > 0).sum().item()} / {concept_vector.shape[1]}")
print(f"Min (non-zero): {concept_vector[concept_vector > 0].min().item():.4f}")
print(f"Max: {concept_vector.max().item():.4f}")
print(f"Mean (non-zero): {concept_vector[concept_vector > 0].mean().item():.4f}")

print(f"\n{'=' * 60}")
print("CONCEPT MAPS (spatial)")
print("=" * 60)
print(f"Shape: {concept_maps.shape}")

# Top active concepts
top_k = 20
top_values, top_indices = concept_vector[0].topk(top_k)
print(f"\nTop {top_k} active concepts:")
for i, (idx, val) in enumerate(zip(top_indices, top_values)):
    name = concept_names[idx.item()] if concept_names else f"concept_{idx.item()}"
    print(f"  {i+1:2d}. [{idx.item():4d}] {name:30s} = {val.item():.4f}")

## Manifold Smoothing in Concept Space
Generate concept vectors for a dataset, build KNN index, then apply
PCA-based manifold smoothing on concept vectors (analogous to Jonas's pixel-space pipeline).

In [ ]:
# Load probe dataset and linear classifier weights
from cfm import config as cfm_config
from cfm.utils import get_probe_dataset
from cfm.method_utils import MethodCFM

PROBE_DATASET = "imagenet"  # or "places365"
PROBE_SPLIT = "val"

args.probe_dataset = PROBE_DATASET
args.probe_split = PROBE_SPLIT
args.probe_dataset_root_dir = cfm_config.probe_dataset_root_dir_dict[PROBE_DATASET]

probe_val_dataset = get_probe_dataset(
    PROBE_DATASET, PROBE_SPLIT, args.probe_dataset_root_dir, preprocess_fn=preprocess)
print(f"Dataset: {PROBE_DATASET} ({PROBE_SPLIT}), {len(probe_val_dataset)} samples")

# Load linear probe: concept_vector @ classifier_weights.T -> class logits
if not hasattr(args, "autoencoder_input_dim_dict"):
    args.autoencoder_input_dim_dict = cfm_config.autoencoder_input_dim_dict

method_obj = MethodCFM(args, vocab_txt_path=None)

# Probe config: no sparsity, max pooling, no threshold
PROBE_CONFIG = "lr0.0001_bs512_epo50_clCE_spL1_spl0.0max_no_threshold"
probe_base = os.path.join(
    args.probe_cs_save_dir_root, args.sae_dataset,
    args.img_enc_name_for_saving, args.hook_points[0],
    args.config_name, PROBE_DATASET, PROBE_CONFIG, "on_concepts_ckpts")
# Find the checkpoint file
ckpt_file = [f for f in os.listdir(probe_base) if f.endswith(".pt")][0]
ckpt_path = os.path.join(probe_base, ckpt_file)
print(f"Loading probe from: {ckpt_path}")

classifier_weights = method_obj.get_classifier_weights(
    probe_dataset=PROBE_DATASET, checkpoint_save_path=ckpt_path).to(args.device)
print(f"Classifier weights: {classifier_weights.shape}")  # [num_classes, 8192]

In [ ]:
# Generate concept vectors for all images and save
# Each vector is [8192], sparse with ~12 non-zero entries
from torch.utils.data import DataLoader

SAVE_DIR = os.path.join("..", "smoothing_data")
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 64
NUM_WORKERS = 4

loader = DataLoader(probe_val_dataset, batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

all_concept_vectors = []
all_labels = []

with torch.no_grad():
    for batch_idx, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(args.device)
        cvs = cfm_model.get_aggregated_concept_activations(imgs)  # [B, 8192]
        all_concept_vectors.append(cvs.cpu())
        all_labels.append(labels)
        if (batch_idx + 1) % 50 == 0:
            print(f"  Batch {batch_idx+1}/{len(loader)}")

all_concept_vectors = torch.cat(all_concept_vectors, dim=0)  # [N, 8192]
all_labels = torch.cat(all_labels, dim=0)  # [N]

torch.save(all_concept_vectors, os.path.join(SAVE_DIR, f"concept_vectors_{PROBE_DATASET}_{PROBE_SPLIT}.pt"))
torch.save(all_labels, os.path.join(SAVE_DIR, f"labels_{PROBE_DATASET}_{PROBE_SPLIT}.pt"))
print(f"Saved {all_concept_vectors.shape[0]} concept vectors, shape {all_concept_vectors.shape}")
print(f"Non-zero per vector (mean): {(all_concept_vectors > 0).sum(1).float().mean():.1f}")

In [ ]:
# Build KNN index on concept vectors using Annoy
import annoy
import json

CONCEPT_DIM = all_concept_vectors.shape[1]  # 8192
N = all_concept_vectors.shape[0]

knn_index = annoy.AnnoyIndex(CONCEPT_DIM, 'euclidean')

for i in range(N):
    knn_index.add_item(i, all_concept_vectors[i].numpy())

N_TREES = 50
knn_index.build(N_TREES)

index_path = os.path.join(SAVE_DIR, f"knn_concepts_{PROBE_DATASET}_{PROBE_SPLIT}.ann")
knn_index.save(index_path)
print(f"Built Annoy index: {N} vectors, dim={CONCEPT_DIM}, trees={N_TREES}")
print(f"Saved to {index_path}")

In [ ]:
# Manifold smoothing on concept vectors
from sklearn.decomposition import PCA
from cfm.data_utils import probe_classnames

def classify_concept_vector(cv, classifier_weights):
    """cv: [8192] -> predicted class index"""
    logits = cv @ classifier_weights.T
    return logits.argmax().item()

def get_class_name(probe_dataset, class_idx):
    names = probe_classnames.probe_classes_dict[probe_dataset]
    if probe_dataset == "places365":
        return " ".join(names[class_idx].split("/")[2:]).replace("_", " ")
    return names[class_idx]

def get_top_concept_names(cv, concept_names, top_k=10):
    """Return list of (name, value) for top-k active concepts"""
    vals, idxs = torch.topk(torch.tensor(cv), top_k)
    return [(concept_names[i.item()], v.item()) for i, v in zip(idxs, vals)]

# Parameters
K_NEIGHBORS = 500
SCALE_WEIGHT = 0.7
N_SMOOTH_SAMPLES = 100
TARGET_IDCS = [100, 51, 42, 200, 500]

for target_idx in TARGET_IDCS:
    # Original concept vector and classification
    cv_orig = all_concept_vectors[target_idx].numpy()
    label_true = all_labels[target_idx].item()
    pred_orig = classify_concept_vector(
        torch.tensor(cv_orig, device=args.device), classifier_weights)

    # Find K nearest neighbors in concept space
    nn_idcs = knn_index.get_nns_by_item(target_idx, K_NEIGHBORS)
    X_neighbors = np.stack([knn_index.get_item_vector(i) for i in nn_idcs])

    # Recenter around neighborhood mean
    mean_nn = X_neighbors.mean(axis=0)
    X_centered = X_neighbors - mean_nn

    # PCA on the neighborhood (concept manifold structure)
    pca = PCA(n_components=K_NEIGHBORS)
    pca.fit(X_centered)
    ev = pca.explained_variance_
    Vt = pca.components_

    # Whiten the original concept vector
    cv_whitened = (cv_orig - mean_nn) @ Vt.T / np.sqrt(ev)

    # Smoothing: add noise in whitened space, transform back, classify
    smooth_preds = []
    smooth_concepts_overlap = []

    orig_top = set(np.argsort(-cv_orig)[:20])

    for _ in range(N_SMOOTH_SAMPLES):
        noise = np.random.normal(0, SCALE_WEIGHT, size=len(ev))
        cv_noised_whitened = cv_whitened + noise
        # Transform back to concept space
        cv_smoothed = cv_noised_whitened @ (np.sqrt(ev)[:, None] * Vt) + mean_nn

        pred_i = classify_concept_vector(
            torch.tensor(cv_smoothed, dtype=torch.float32, device=args.device),
            classifier_weights)
        smooth_preds.append(pred_i)

        # Track concept overlap
        noised_top = set(np.argsort(-cv_smoothed)[:20])
        overlap = len(orig_top & noised_top) / 20.0
        smooth_concepts_overlap.append(overlap)

    # Majority vote
    from collections import Counter
    vote_counts = Counter(smooth_preds)
    pred_smooth, n_votes = vote_counts.most_common(1)[0]

    # Results
    print("=" * 70)
    print(f"Image idx={target_idx}")
    print(f"  True class:     {get_class_name(PROBE_DATASET, label_true)}")
    print(f"  Original pred:  {get_class_name(PROBE_DATASET, pred_orig)}")
    print(f"  Smoothed pred:  {get_class_name(PROBE_DATASET, pred_smooth)} "
          f"({n_votes}/{N_SMOOTH_SAMPLES} votes)")
    print(f"  Prediction stable: {pred_orig == pred_smooth}")
    print(f"  Mean top-20 concept overlap: {np.mean(smooth_concepts_overlap):.3f}")

    # Show which concepts are active
    orig_top_names = get_top_concept_names(cv_orig, concept_names, top_k=10)
    print(f"  Top 10 concepts (original):")
    for name, val in orig_top_names:
        print(f"    {name:30s} {val:.4f}")

In [ ]:
# Detailed concept comparison: original vs manifold-smoothed (averaged)
import matplotlib.pyplot as plt

TARGET = TARGET_IDCS[0]
cv_orig = all_concept_vectors[TARGET].numpy()
label_true = all_labels[TARGET].item()

nn_idcs = knn_index.get_nns_by_item(TARGET, K_NEIGHBORS)
X_neighbors = np.stack([knn_index.get_item_vector(i) for i in nn_idcs])
mean_nn = X_neighbors.mean(axis=0)
X_centered = X_neighbors - mean_nn
pca = PCA(n_components=K_NEIGHBORS)
pca.fit(X_centered)
ev = pca.explained_variance_
Vt = pca.components_
cv_whitened = (cv_orig - mean_nn) @ Vt.T / np.sqrt(ev)

# Average many smoothed samples to get stable manifold concept vector
N_AVG = 100
smoothed_samples = []
for _ in range(N_AVG):
    noise = np.random.normal(0, SCALE_WEIGHT, size=len(ev))
    cv_smoothed = (cv_whitened + noise) @ (np.sqrt(ev)[:, None] * Vt) + mean_nn
    smoothed_samples.append(cv_smoothed)
cv_manifold_avg = np.mean(smoothed_samples, axis=0)

# Get scores
orig_scores = cv_orig.copy()
mani_scores = cv_manifold_avg.copy()

TOP_K = 15

# --- 1) Top Activated: highest score in ORIGINAL, show both ---
top_orig_idxs = np.argsort(-orig_scores)[:TOP_K]
top_orig_vals = orig_scores[top_orig_idxs]
top_mani_vals = mani_scores[top_orig_idxs]
top_names = [concept_names[i] for i in top_orig_idxs]

# --- 2) Least Activated: lowest score in MANIFOLD (among originally active), show both ---
active_mask = orig_scores > 0  # only consider concepts that were active somewhere
mani_active = mani_scores.copy()
mani_active[~active_mask] = np.inf  # exclude inactive
least_mani_idxs = np.argsort(mani_active)[:TOP_K]
least_mani_vals = mani_scores[least_mani_idxs]
least_orig_vals = orig_scores[least_mani_idxs]
least_names = [concept_names[i] for i in least_mani_idxs]

# --- 3) Top Drops: largest drop from original to manifold ---
drops = orig_scores - mani_scores
drop_idxs = np.argsort(-drops)[:TOP_K]
drop_orig_vals = orig_scores[drop_idxs]
drop_mani_vals = mani_scores[drop_idxs]
drop_names = [concept_names[i] for i in drop_idxs]

# Shared scale: use the global max across all values shown
global_max = max(orig_scores.max(), mani_scores.max()) * 1.05

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 8))
bar_height = 0.35
y_pos = np.arange(TOP_K)

# --- Plot 1: Top Activated (by original score) ---
ax = axes[0]
ax.barh(y_pos + bar_height/2, top_orig_vals[::-1], bar_height, label='Original', color='#2196F3')
ax.barh(y_pos - bar_height/2, top_mani_vals[::-1], bar_height, label='Manifold', color='#FF9800')
ax.set_yticks(y_pos)
ax.set_yticklabels(top_names[::-1], fontsize=8)
ax.set_xlim(0, global_max)
ax.set_xlabel('Activation Score')
ax.set_title('Top Activated Concepts\n(ranked by original score)')
ax.legend(loc='lower right')

# --- Plot 2: Least Activated after Manifold ---
ax = axes[1]
ax.barh(y_pos + bar_height/2, least_orig_vals[::-1], bar_height, label='Original', color='#2196F3')
ax.barh(y_pos - bar_height/2, least_mani_vals[::-1], bar_height, label='Manifold', color='#FF9800')
ax.set_yticks(y_pos)
ax.set_yticklabels(least_names[::-1], fontsize=8)
ax.set_xlim(0, global_max)
ax.set_xlabel('Activation Score')
ax.set_title('Least Activated after Manifold\n(lowest manifold score among active concepts)')
ax.legend(loc='lower right')

# --- Plot 3: Top Drops (original had activation, dropped in manifold) ---
ax = axes[2]
ax.barh(y_pos + bar_height/2, drop_orig_vals[::-1], bar_height, label='Original', color='#2196F3')
ax.barh(y_pos - bar_height/2, drop_mani_vals[::-1], bar_height, label='Manifold', color='#FF9800')
ax.set_yticks(y_pos)
ax.set_yticklabels(drop_names[::-1], fontsize=8)
ax.set_xlim(0, global_max)
ax.set_xlabel('Activation Score')
ax.set_title('Top Drops\n(largest decrease: original → manifold)')
ax.legend(loc='lower right')

pred_orig = classify_concept_vector(
    torch.tensor(cv_orig, device=args.device), classifier_weights)
pred_mani = classify_concept_vector(
    torch.tensor(cv_manifold_avg, dtype=torch.float32, device=args.device), classifier_weights)

plt.suptitle(
    f"Concept Comparison: Original vs Manifold Smoothed (idx={TARGET})\n"
    f"True: {get_class_name(PROBE_DATASET, label_true)} | "
    f"Orig pred: {get_class_name(PROBE_DATASET, pred_orig)} | "
    f"Manifold pred: {get_class_name(PROBE_DATASET, pred_mani)}",
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("concept_comparison.png", dpi=150, bbox_inches="tight")
plt.show()